# 0 Fine tuning and training

Trains every model behind the results in `results/`. Notebooks 1 to 3 read those results and do not need this notebook.

| Step | Output in `results_retrained/` | Time on one GPU |
|---|---|---|
| Stage 1: normal behavior models | `stage1/` | about an hour |
| Stage 2: hyperparameter tuning | `stage2/tuned_hyperparameters.json` | about a day |
| Stage 2: Cases 1 to 9, five repeats | `stage2/case_results.csv` | several hours |
| Stage 2: Case 5 for Fig. 14 | `stage2/case5_*` | minutes |
| Monte Carlo dropout settings | `stage2/mc_*.csv` | several hours |
| Draws for Figs. 8 and 9 | `stage2/mc_window_draws.npz` | minutes |

Every step is switched off by default. Setting its flag to `True` trains it again and writes the result to `results_retrained/`. The stored results of the paper in `results/` are never changed. A step that is not retrained uses the stored results, so retraining only stage 2 uses the stored stage 1 residuals and tuned hyperparameters. Notebooks 1 to 3 read the retrained results when their setting `RESULTS` is `'retrained'`.

Stage 2 uses the stage 1 residuals as input, so after retraining stage 1, stage 2 should be retrained too. The Case 5 run selects its models from the case results, so it follows the case runs.

**Reproducibility.** Seeds are fixed, but some GPU operations are not deterministic, so a rerun matches the stored results closely but not bit for bit. A single run of one case can differ noticeably, while the averages over cases and repeats reported in the paper change little. The stored results were also produced before a correction of the one-hour forward fill: missing 10-minute records were left out of the files instead of written as empty rows, so the fill had no effect. The code now puts them back before filling. This leaves the stage 2 windows and RUL labels unchanged and shifts the inputs slightly.

In [1]:
import json

import numpy as np
import pandas as pd

from two_stage import config as cfg, data, evaluation, experiments, stage1, stage2, tuning

cfg.use_results('retrained')

TRAIN_STAGE1 = False
TUNE = False
RUN_CASES = False
RUN_CASE5 = False
RUN_MC_SETTINGS = False
RUN_MC_DRAWS = False

print('device:', cfg.DEVICE)

device: cuda


## Stage 1: normal behavior models

One LSTM normal behavior model per event and component (30 in total), trained on the normal-status records of the training period. The residuals over both periods are stored per model.

In [2]:
info = data.load_info()
features = data.load_feature_description()
runs = stage1.runs(info)
if TRAIN_STAGE1:
    for run in runs:
        residuals, meta = stage1.train_case(run['event'], run['component'], features)
        stage1.save_case(residuals, meta)
        print(f"event {run['event']:>2} {run['component']:<11} validation MAPE {meta['val_mape']:.2f}%")

## Stage 2: hyperparameter tuning

20 Optuna trials per network on Case 1 (T1 and B1 held out). Each trial trains three independently seeded ensembles and scores their mean validation CRPS, with early stopping on the same mean. The bin size and window length are tuned with the network. The Constant baseline uses 8-hour bins and a 14-day window.

In [3]:
if TUNE:
    tuned = {model: tuning.tune(model) for model in stage2.MODELS}
    cfg.output_path('stage2', 'tuned_hyperparameters.json').write_text(json.dumps(tuned, indent=1))
tuned = evaluation.load_tuned()
pd.DataFrame(tuned).T[['resample_freq', 'window_days', 'window', 'epochs', 'score']]

,resample_freq,window_days,window,epochs,score
lstm+none,1h,28.0,672,19,5.223139
lstm+embed,1h,7.0,168,13,5.539039
lstm+head_linear,24h,28.0,28,17,6.457402
lstm+head_mlp,24h,28.0,28,11,5.931649
lstm+separate,8h,28.0,84,21,6.054946
cnn+none,1h,28.0,672,22,6.008928
cnn+embed,1h,28.0,672,8,5.361413
cnn+head_linear,1h,14.0,336,1,4.47455
cnn+head_mlp,1h,28.0,672,15,3.503441
cnn+separate,8h,14.0,42,8,6.313755


## Stage 2: Cases 1 to 9

Every model is trained on the four training failures of each case and scored on the two held-out failures, five times with different seeds. The scores are saved after each model, and a model that is already saved is skipped, so an interrupted run continues where it stopped. To start over, remove `results_retrained/stage2/case_results.csv`.

In [4]:
if RUN_CASES:
    experiments.case_results(tuned, cfg.output_path('stage2', 'case_results.csv'))
evaluation.load_case_results().groupby('model').size()

model
cnn+embed               90
cnn+head_linear         90
cnn+head_mlp            90
cnn+none                90
cnn+separate            90
cnn_lstm+embed          90
cnn_lstm+head_linear    90
cnn_lstm+head_mlp       90
cnn_lstm+none           90
cnn_lstm+separate       90
constant                90
lstm+embed              90
lstm+head_linear        90
lstm+head_mlp           90
lstm+none               90
lstm+separate           90
dtype: int64

## Case 5 for Fig. 14

The most accurate model of each group and the Constant baseline, trained again on Case 5 with the seeds of their median repeat, keeping the predictions over time.

In [5]:
if RUN_CASE5:
    arrays, scores = experiments.case_predictions(tuned, evaluation.load_case_results())
    np.savez_compressed(cfg.output_path('stage2', 'case5_predictions.npz'), **arrays)
    scores.to_csv(cfg.output_path('stage2', 'case5_scores.csv'), index=False)
pd.read_csv(cfg.result_path('stage2', 'case5_scores.csv'))

,group,model,repeat,event,held_out,n,mae,crps,r2,cov95,width95
0,specialized,cnn_lstm+embed,2,7,True,768,5.754363,4.561143,0.467425,0.639323,14.675564
1,specialized,cnn_lstm+embed,2,27,True,1497,16.729190,13.290776,-0.217312,0.332665,23.858366
2,specialized,cnn_lstm+embed,2,34,False,528,1.473972,1.178123,0.929142,0.990530,10.128720
3,specialized,cnn_lstm+embed,2,53,False,1009,0.756037,0.951492,0.992227,1.000000,13.267912
4,specialized,cnn_lstm+embed,2,19,False,504,0.857805,1.007899,0.968559,0.994048,10.366169
5,specialized,cnn_lstm+embed,2,77,False,1475,1.130775,1.330036,0.993281,1.000000,17.242367
6,independent,cnn+separate,1,7,True,96,7.627587,5.926208,0.018893,0.531250,15.806424
7,independent,cnn+separate,1,27,True,188,13.601451,11.576193,0.078254,0.414894,17.048556
8,independent,cnn+separate,1,34,False,66,2.689567,1.875008,0.741422,0.939394,13.587151
9,independent,cnn+separate,1,53,False,127,5.022486,4.285795,0.602531,0.771654,17.445265


## Monte Carlo dropout settings

Two checks of the predictive distribution: the number of passes per network (5 to 1,000 passes, on Case 1) and the number of networks per ensemble (the 15 networks of the five repeats of every case, pooled into ensembles of 1 to 15 networks).

In [6]:
if RUN_MC_SETTINGS:
    experiments.passes_per_network(tuned).to_csv(cfg.output_path('stage2', 'mc_passes.csv'), index=False)
    experiments.ensemble_size(tuned).to_csv(cfg.output_path('stage2', 'mc_ensemble_size.csv'), index=False)
evaluation.load_ensemble_size().groupby('networks')[['crps', 'cov95', 'width95']].median()

,crps,cov95,width95
networks,,,
1,9.292914,0.377049,14.128775
2,8.020258,0.514648,18.478186
3,7.967760,0.562767,21.056693
4,7.621927,0.619673,22.214829
5,7.659191,0.632772,23.215779
6,7.648844,0.644847,23.615677
8,7.475536,0.671908,24.741607
10,7.472315,0.683084,25.372280
12,7.439711,0.692913,25.492896


## Draws for Figs. 8 and 9

The independent CNN models on Case 5. The window whose spread between the three networks is the median over both held-out failures is kept.

In [7]:
if RUN_MC_DRAWS:
    np.savez(cfg.output_path('stage2', 'mc_window_draws.npz'), **experiments.window_draws(tuned))
draws = np.load(cfg.result_path('stage2', 'mc_window_draws.npz'))
print(f"event {int(draws['event'])}, window {int(draws['window_index'])}, mean {float(draws['point']):.1f} d")

event 7, window 79, mean 8.7 d


## Retrained results

In [8]:
retrained = cfg.RESULTS_DIRS['retrained']
sorted(str(p.relative_to(retrained)) for p in retrained.rglob('*') if p.is_file()) if retrained.exists() else []

[]